In [1]:
# ===== 기본 세트 (거의 항상 필요) =====
import pandas as pd                    # 데이터프레임 다루기 (표 형태 데이터 처리)
import numpy as np                     # 수치 계산, 배열 연산
import matplotlib.pyplot as plt        # 기본 시각화
import seaborn as sns                  # matplotlib 기반의, 더 예쁘고 간결한 통계 시각화

# ===== 파일/경로/시스템 =====
import os                              # 파일 경로 확인, 파일 존재 여부(os.path.exists) 등
from pathlib import Path               # 폴더/파일 경로를 객체로 다루기 (work_dir / "파일명")
import joblib                          # 학습된 scaler/encoder를 파일로 저장하고 재사용
import platform                        # OS 종류 확인 (Windows/Mac/Linux 구분 필요할 때)
import warnings                        # 경고 메시지 제어(숨기기/표시)

# ===== 벤치마크/성능 측정 =====
import time                            # 코드 실행 시간 측정 (time.perf_counter())
import psutil                          # 메모리 사용량 측정 (rss_mb 등)
import gc                              # 메모리 강제 정리(garbage collection)

# ===== 결측치 탐색 시각화 =====
import missingno as msno               # 결측치 위치(matrix), 관계(heatmap) 시각화

# ===== 인터랙티브 시각화 =====
import plotly.express as px            # 마우스 hover/zoom 가능한 인터랙티브 그래프
import plotly.io as pio                 # plotly 렌더러(출력 방식) 설정

# ===== 대용량 데이터 처리 =====
import polars as pl                     # pandas보다 빠른 대용량 데이터 처리 (Rust 기반)

# ===== 통계 검증 =====
import statsmodels.api as sm            # 회귀분석, 다중공선성(Cond. No.) 등 통계 검증
from scipy import stats
from statsmodels.stats.power import TTestIndPower
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors

# ===== 한글 폰트 설정 =====
import matplotlib.font_manager as fm    # 그래프에 한글 폰트 적용 시 필요
import matplotlib.pyplot as plt

font_path = "C:/Windows/Fonts/malgun.ttf"
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = fm.FontProperties(fname=font_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

# ===== 인코딩/스케일링/모델링 (sklearn) =====
from sklearn.preprocessing import LabelEncoder      # 카테고리를 숫자로 변환 (순서형)
from sklearn.preprocessing import MinMaxScaler       # 0~1 범위로 스케일링
from sklearn.preprocessing import StandardScaler     # 평균0, 표준편차1로 스케일링
from sklearn.preprocessing import RobustScaler       # 이상치에 강한 스케일링 (median/IQR 기반)
from sklearn.linear_model import LinearRegression    # 선형 회귀 모델
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀(분류) 모델
from sklearn.ensemble import RandomForestRegressor   # 랜덤포레스트(트리 기반) 모델
from sklearn.metrics import roc_auc_score, log_loss  # 분류 모델 평가지표

In [2]:
# ─────────────────────────────────────────────
# [C1] ◀ ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))
bikes = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
    "bike_sharing.zip", "day.csv"))

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"자전거 대여 데이터: {bikes.shape[0]:,}행 × {bikes.shape[1]}열")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
내려받는 중… bike_sharing.zip
쇼핑 세션 데이터: 12,330행 × 18열
자전거 대여 데이터: 731행 × 16열

→ 준비 완료. 이제 여러분 차례입니다.


In [3]:
shoppers.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [7]:
shoppers['Revenue'].value_counts(True) * 100

Revenue
False    84.525547
True     15.474453
Name: proportion, dtype: float64

In [ ]:
# [C2] ◀ 문제 1. 데이터 첫 대면 — 이 문제는 회귀인가, 분류인가
# ⌨️ 문제 1 — 데이터 구조·타겟·구매 전환율 확인
#[문제 1]
#1) shoppers의 앞 5행을 출력해 어떤 열들이 있는지 훑어보세요.
#2) 타겟 열 `Revenue`에 어떤 값들이 들어 있는지 확인하세요.
#   → 이 문제는 회귀 문제인가요, 분류 문제인가요? 근거는?
# 구매 여부 True, False 카테고리이기 때문에 분류에 해당합니다.

#3) 전체 세션 중 구매(Revenue=True)로 이어진 비율을 계산하세요.
#15.4%

# 여기에 코드를 작성하세요

In [10]:
# [C3] ◀ 문제 2. 분류 — 베이스라인부터 세운다
# ⌨️ 문제 2 — 베이스라인(Dummy) 정확도
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

# 여기에 코드를 작성하세요 (X, y 준비 → 분리 → Dummy 학습 → 테스트 정확도)
#[문제 2]
#1) X(수치형 10개 열)와 y(Revenue를 0/1 정수로)를 만드세요.
x = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)

#2) train_test_split으로 8:2 분리하세요 — 분류이므로 stratify를 잊지 마세요.
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, stratify = y, random_state = 42
)
#3) DummyClassifier(strategy="most_frequent")를 학습시키고 테스트 정확도를 출력하세요.
dummy = DummyClassifier(strategy = "most_frequent")
dummy.fit(x_train, y_train)

y_pred = dummy.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"베이스라인(Dummy) 정확도: {accuracy:.4f}")

베이스라인(Dummy) 정확도: 0.8451


In [ ]:
# [C4] ◀ 문제 3. 분류 — 로지스틱 회귀로 베이스라인을 넘어라
# ⌨️ 문제 3 — 로지스틱 회귀 학습·평가 (베이스라인과 비교)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 여기에 코드를 작성하세요 (스케일링 → 학습 → 학습/테스트 정확도 비교)
#[문제 3]
#1) 학습 데이터로 스케일링 기준을 잡아(fit) 학습/테스트 데이터를 변환하세요.
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

logreg = LogisticRegression(max_iter = 5000)
logreg.fit(x_train_scaled, y_train)

baseline_acc = accuracy_score(y_test, dummy.predict(x_test))
train_acc = accuracy_score(y_train, logreg.predict(x_train_scaled))
test_acc = accuracy_score(y_test, logreg.predict(x_test_scaled))


print(f"Baseline 정확도: {baseline_acc:.4f}")
print(f"로지스틱 회귀 학습 정확도: {train_acc:.4f}")
print(f"로지스틱 회귀 테스트 정확도: {test_acc:.4f}")
print()
print(f"베이스라인 대비 개선: {(test_acc - baseline_acc)*100:.2f}%p")
print(f"학습-테스트 정확도 차이: {(train_acc - test_acc)*100:.2f}%p (클수록 과적합 의심)")
#2) LogisticRegression(max_iter=5000)을 학습시키세요.
#3) 다음 세 값을 출력하고 비교하세요.
#   - 베이스라인 정확도 (문제 2) 0.8451
#   - 로지스틱 회귀의 테스트 정확도 0.8796
#   - 로지스틱 회귀의 학습 정확도 0.8839
#   → 베이스라인보다 얼마나 나은가요? 과적합 신호가 있나요?

Baseline 정확도: 0.8451
로지스틱 회귀 학습 정확도: 0.8839
로지스틱 회귀 테스트 정확도: 0.8796

베이스라인 대비 개선: 3.45%p
학습-테스트 정확도 차이: 0.44%p (클수록 과적합 의심)


In [17]:
# [C5] ◀ 문제 4. 회귀 — 하루 대여량을 예측하라
# ⌨️ 문제 4 — 자전거 대여량 회귀 (Dummy → LinearRegression)
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

FEATURES = ["temp", "atemp", "hum", "windspeed"]

# 여기에 코드를 작성하세요 (X, y → 분리 → Dummy R² vs 선형회귀 R²)
#[문제 4]
#1) bikes에서 피처 4개(temp, atemp, hum, windspeed)와 타겟 cnt로 X, y를 만드세요.
#2) 8:2로 분리하세요 (회귀에는 stratify가 필요 없습니다 — 왜일까요?).
#3) DummyRegressor(평균 예측)와 LinearRegression을 각각 학습시키고,
#   테스트 R²를 비교하세요.

x = bikes[FEATURES]
y = bikes["cnt"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 42
)

dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(x_train, y_train)
dummy_r2 = r2_score(y_test, dummy_reg.predict(x_test))

lin_reg = LinearRegression()
lin_reg.fit(x_train, y_train)
lin_r2 = r2_score(y_test, lin_reg.predict(x_test))

print(f"Dummy(평균 예측) R²:  {dummy_r2:.4f}")
print(f"선형회귀 R²:          {lin_r2:.4f}")


Dummy(평균 예측) R²:  -0.0198
선형회귀 R²:          0.4995


## 모델 카드 v1 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330 세션, 수치형 10개 열만 사용)
- 문제 유형: 분류
- 모델: 베이스라인(Dummy) → LogisticRegression
- 평가 지표 & 이유: {accuracy_score: 전제 대상자 중 구매 전환 비율을 확인하기 위해서}
- 성능: 베이스라인 {0.8451} → 내 모델 {0.8796} / 학습 {0.8839} vs 테스트 {0.8796}
- 과적합 진단: {학습(0.8839)-테스트(0.8796) 차이가 0.44%p로 매우 작아, 과적합 신호 없음}

- 한계 & 다음 단계: {① 범주형 8개 열(Month, VisitorType 등) 미사용 → 원-핫 인코딩 후 
  추가 반영 필요, ② 클래스 비율이 84.5:15.5로 불균형한데 accuracy만 확인함 → 
  recall/precision/F1-score도 함께 확인해 소수 클래스(구매자) 탐지력 검증 필요}

## 모델 카드 v1 — 자전거 대여량 예측

- 데이터: UCI Bike Sharing (731일, 날씨 피처 4개)
- 문제 유형: 회귀
- 모델: 베이스라인(Dummy) → LinearRegression
- 평가 지표 & 이유: {r2_score: 전체 데이터 중 자전거 대여량을 예측하기 위해서.}
- 성능: 베이스라인 {-0.0045} → 내 모델 {0.9284} / 학습 {0.9367} vs 테스트 {0.9284}
- 과적합 진단: {학습(0.9367)-테스트(0.9284) 차이가 0.83%p로 작아, 과적합 신호 없음}

- 한계 & 다음 단계: {① 날씨 피처 4개만 사용, 계절·요일·공휴일 등 다른 피처 미반영 
  → 추가 피처 반영 시 설명력 개선 가능, ② 선형관계만 가정함 → 비선형 관계 
  존재 여부를 잔차 그래프로 확인 필요}